In [ ]:
import pandas as pd

In [ ]:
data_path = "/mnt/data1/klug/datasets/meditron/Meditron_ICU.xlsx"

In [ ]:
df = pd.read_excel(data_path)

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
first_answer_df = df[['First Answer', 'Question', 'Name', 
                      'Eval Alignment with Guidelines', 'Eval Question Comprehension',
       'Eval Logical Reasoning', 'Eval Relevance & Completeness',
       'Eval Harmlessness', 'Eval Fairness', 'Eval Contextual Awareness',
       'Eval Your Confidence', 'Eval Model Confidence',
       'Eval Communication & Clarity']]
first_answer_df.rename(columns={'First Answer': 'Answer'}, inplace=True)

second_answer_df = df[['Second Answer', 'Question', 'Name', 
                       'Second Eval Alignment with Guidelines',
       'Second Eval Question Comprehension', 'Second Eval Logical Reasoning',
       'Second Eval Relevance & Completeness', 'Second Eval Harmlessness',
       'Second Eval Fairness', 'Second Eval Contextual Awareness',
       'Second Eval Your Confidence', 'Second Eval Model Confidence',
       'Second Eval Communication & Clarity']]
second_answer_df.rename(columns={'Second Answer': 'Answer', 
                                 'Second Eval Alignment with Guidelines': 'Eval Alignment with Guidelines',
       'Second Eval Question Comprehension': 'Eval Question Comprehension',
       'Second Eval Logical Reasoning': 'Eval Logical Reasoning',
       'Second Eval Relevance & Completeness': 'Eval Relevance & Completeness',
       'Second Eval Harmlessness': 'Eval Harmlessness',
       'Second Eval Fairness': 'Eval Fairness',
       'Second Eval Contextual Awareness': 'Eval Contextual Awareness',
       'Second Eval Your Confidence': 'Eval Your Confidence',
       'Second Eval Model Confidence': 'Eval Model Confidence',
       'Second Eval Communication & Clarity': 'Eval Communication & Clarity'}, inplace=True)

all_answers_df = pd.concat([first_answer_df, second_answer_df], ignore_index=True)

In [ ]:
domains = ['Eval Alignment with Guidelines', 'Eval Question Comprehension',
       'Eval Logical Reasoning', 'Eval Relevance & Completeness',
       'Eval Harmlessness', 'Eval Fairness', 'Eval Contextual Awareness',
       'Eval Your Confidence', 'Eval Model Confidence',
       'Eval Communication & Clarity']

In [ ]:
import krippendorff

#  get agreement across domains: both in terms of std of answers and krippendorf alpha
answer_eval_agreement_std = pd.DataFrame(columns=['Answer', 'Question'] + domains)
answer_eval_agreement_krippendorf = pd.DataFrame(columns=['Answer', 'Question'] + domains)

for answer in all_answers_df['Answer'].unique():
    answer_df = all_answers_df[all_answers_df['Answer'] == answer]
    question = answer_df['Question'].iloc[0]
    row_std = {'Answer': answer, 'Question': question}
    row_alpha = {'Answer': answer, 'Question': question}
    for domain in domains:
        domain_std = answer_df[domain].std()
        row_std[domain] = domain_std
        # krippendorff alpha
        data = answer_df[domain].values
        # if only one non nan value, set alpha to nan
        if len(data[~pd.isna(data)]) <= 1:
            domain_alpha = float('nan')
        # elif all values are the same, set alpha to 1
        elif len(set(data[~pd.isna(data)])) == 1:
            domain_alpha = 1.0
        else:
            domain_alpha = krippendorff.alpha(reliability_data=data, level_of_measurement='ordinal', value_domain=[0, 1, 2, 3, 4, 5])
        row_alpha[domain] = domain_alpha
        
    answer_eval_agreement_std = pd.concat([answer_eval_agreement_std, pd.DataFrame([row_std])], ignore_index=True)
    answer_eval_agreement_krippendorf = pd.concat([answer_eval_agreement_krippendorf, pd.DataFrame([row_alpha])], ignore_index=True)
    

In [ ]:
answer_eval_agreement_std['Eval Alignment with Guidelines'].describe()